# Bronze Layer — Customers Ingestion via Autoloader (Mounting Version)
**GlobalMart | Tredence DE Advanced Training**

| | |
|---|---|
| **Source** | Blob Storage container, mounted via `dbutils.fs.mount()` → `raw-data/customers/` |
| **Mount source** | `wasbs://YOUR_CONTAINER@YOUR_STORAGE_ACCOUNT.blob.core.windows.net/` |
| **Target** | `YOUR_CATALOG.bronze.customers` (Managed Delta Table) |
| **Auth** | Storage account access key, set once at mount time — no Unity Catalog External Location involved |
| **Mode** | `trigger(availableNow=True)` — processes all new files, then stops |

### How this differs from the External-Location version of this same task

| | External Location version | This (Mounting) version |
|---|---|---|
| Connection | Unity Catalog Storage Credential + External Location | `dbutils.fs.mount()` with a storage account key |
| Protocol | `abfss://` + `.dfs.core.windows.net` (requires ADLS Gen2 / Hierarchical Namespace) | `wasbs://` + `.blob.core.windows.net` |
| Catalog | `<your-catalog>` (same catalog as the External Location walkthrough) | A separate practice catalog — different from the one above |
| Checkpoint/schema location | Shared root `_checkpoints/<table>/` and `_schemas/<table>/` folders | Nested **inside** each table's own source folder — `raw-data/customers/_checkpoints/` |

> **Why `wasbs://` and not `abfss://` here:** `abfss://` only works for mounting/reading against a storage account with **Hierarchical Namespace (ADLS Gen2)** enabled. If your storage account doesn't have that enabled, or your workspace's mount setup doesn't support `abfss://` mounting, `wasbs://` against the `.blob.core.windows.net` endpoint is the working alternative — same underlying data, older Blob Storage protocol.

> **Why bother with this at all, if Day 2 called mounting a legacy anti-pattern?** Day 2 was right that nobody sets up a *new* mount for a governed, ongoing pipeline anymore — External Locations exist precisely to replace it. But plenty of real workspaces still have old mounts sitting around from before that shift, and you'll be expected to recognize the pattern, know its trade-offs, and migrate off it when asked. That's why it's still worth building once, hands-on, even though it's not where you'd start a brand-new pipeline today.

## Step 0 — Mount the Container

Unlike the External Location version, this notebook reaches storage the old way: a real storage account key, set once via `dbutils.fs.mount()`. Get your key first:

```
Azure Portal → Storage accounts → YOUR_STORAGE_ACCOUNT
  → Security + networking → Access keys
  → click "Show" next to key1 → Copy
```

Paste it into the cell below, run it, then replace it back with the placeholder before saving/sharing this notebook — never leave a real key in a notebook anyone else can open.

In [ ]:
# Storage Details
STORAGE_ACCOUNT = "ecomdata"
CONTAINER = "raw-demo-data"
MOUNT_POINT = "/mnt/virinchy_gbmart_data"

STORAGE_ACCOUNT_KEY = "YOUR_STORAGE_ACCOUNT_KEY"  # ← paste your own key here, never commit a real one

configs = {
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.blob.core.windows.net": STORAGE_ACCOUNT_KEY
}

# Check if already mounted
if any(m.mountPoint == MOUNT_POINT for m in dbutils.fs.mounts()):
    print(f"✅ {MOUNT_POINT} is already mounted.")
else:
    dbutils.fs.mount(
        source=f"wasbs://{CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net/",
        mount_point=MOUNT_POINT,
        extra_configs=configs
    )
    print(f"✅ Successfully mounted at {MOUNT_POINT}")

In [ ]:
# ─── Verify the mount worked ───────────────────────────────────────────────────
print(f"Contents of {MOUNT_POINT} :")
for f in dbutils.fs.ls(MOUNT_POINT):
    print(f"  {f.name}")

## Step 1 — Configuration

**The important change from the External Location version:** checkpoint and schema paths are **not** in a shared root `_checkpoints/` / `_schemas/` folder with one subfolder per table. They live **inside each table's own source folder** instead — `raw-data/customers/_checkpoints/`, not `raw-data/_checkpoints/customers/`.

```
OLD (shared root folders):                 NEW (nested inside each table's own folder):
raw-data/                                  raw-data/
  _checkpoints/                              customers/
    customers/          ← shared root          customers_010626.csv
    orders/                                    _checkpoints/     ← nested here
  _schemas/                                    _schemas/         ← nested here
    customers/                               orders/
    orders/                                    ...
```

> **One thing worth knowing before you commit to this layout:** having Autoloader's own checkpoint/schema state sit inside the exact folder it's scanning for source files is a slightly unusual pattern — most Databricks reference architectures keep those state folders completely separate from the data folder to avoid any chance of the listing operation touching its own metadata. In practice this works because `cloudFiles.format="csv"` only picks up files that look like CSVs, so the JSON/state files Autoloader writes under `_checkpoints/`/`_schemas/` won't get ingested as data — but it's worth being aware this is a deliberate, non-default choice.

In [ ]:
from pyspark.sql.functions import input_file_name, current_timestamp

# Mounted base path — set once, reuse everywhere
MOUNT_BASE = f"{MOUNT_POINT}"

SOURCE_FOLDER = "customers"

CATALOG = "harsh_kumar01_npmentorskool_onmicrosoft_com"   # ← different catalog than gbmart
SCHEMA = "bronze"
TABLE = "customers"
TARGET_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE}"

# Source data path
SOURCE_PATH = f"{MOUNT_BASE}/{SOURCE_FOLDER}/"

# Auto Loader metadata paths
# Each table gets its own checkpoint and schema subfolder.
CHECKPOINT_PATH = f"{MOUNT_BASE}/_checkpoints/{SOURCE_FOLDER}/"
SCHEMA_PATH = f"{MOUNT_BASE}/_schemas/{SOURCE_FOLDER}/"

print(f"Source      : {SOURCE_PATH}")
print(f"Target table: {TARGET_TABLE}")
print(f"Checkpoint  : {CHECKPOINT_PATH}")
print(f"Schema      : {SCHEMA_PATH}")

## Step 2 — Verify Files in the Mounted Source Folder

In [ ]:
files = dbutils.fs.ls(SOURCE_PATH)
print(f"Files found in {SOURCE_FOLDER}/:\n")
for f in files:
    print(f"  {f.name}  ({f.size / 1024:.1f} KB)")

## Step 3 — Create Catalog & Schema (if not exists)

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"Catalog '{CATALOG}' and schema '{SCHEMA}' are ready.")

## Step 4 — Autoloader Ingestion

| Option | Value | Why |
|---|---|---|
| `cloudFiles.format` | `csv` | source file format |
| `cloudFiles.schemaLocation` | `SCHEMA_PATH` (nested under `customers/_schemas/`) | saves inferred schema — reused on next run |
| `cloudFiles.inferColumnTypes` | `true` | infers proper types instead of all string |
| `cloudFiles.schemaEvolutionMode` | `addNewColumns` | new columns in future files are added automatically |
| `mergeSchema` | `true` | Delta write-side schema merge — resolves schema mismatch on first encounter |
| `trigger(availableNow)` | — | batch-style: process all new files then stop |

In [ ]:
from pyspark.sql.functions import *

In [ ]:
customers_df = spark.readStream\
        .format("cloudFiles")\
        .option("cloudFiles.format",              "csv")\
        .option("cloudFiles.schemaLocation",      SCHEMA_PATH)\
        .option("cloudFiles.inferColumnTypes",    "true")\
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")\
        .option("header",                         "true")\
        .load(SOURCE_PATH)\
        .withColumn("_source_file", col("_metadata.file_path"))\
        .withColumn("_ingested_at", current_timestamp())

In [ ]:
customers_df.display()

In [ ]:
customers_df.writeStream\
        .format("delta")\
        .outputMode("append")\
        .option("checkpointLocation", CHECKPOINT_PATH)\
        .option("mergeSchema",        "true")\
        .trigger(availableNow=True)\
        .toTable(TARGET_TABLE)

## Step 5 — Verify Data in Bronze Table

In [ ]:
df = spark.table(TARGET_TABLE)
print(f"Total rows : {df.count()}")
print(f"Columns    : {df.columns}")
df.display(5, truncate=False)

In [ ]:
# Row count per source file
df.groupBy("_source_file").count().orderBy("_source_file").display(truncate=False)